# 📘 Tutorial 4: Practical Modelling Choices in BoTorch

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Tutorial 3**, we built the full sequential Bayesian Optimisation loop in BoTorch.

We saw that, once an initial dataset is available, we can repeatedly:

- fit a Gaussian Process surrogate,
- construct an acquisition function from that surrogate,
- optimise the acquisition function to choose the next point,
- evaluate the objective there,
- update the dataset,
- and continue the BO process over multiple iterations.

That established the core structure of standard Bayesian Optimisation as a working algorithm.

In this tutorial, we take the next practical step:

> **once the standard BO loop is in place, which modelling choices most strongly affect how well it behaves in practice?**

That is the focus of this notebook.

This is a different kind of shift from the previous tutorial.

In Tutorial 3, the emphasis was on understanding the BO loop itself:
how the surrogate, acquisition function, and dataset update fit together as a repeated optimisation process.

Here, the BO loop remains the same.

What changes is the set of **practical choices around that loop**:

- whether inputs are normalised,
- whether outputs are standardised,
- how noisy the observations are,
- how carefully the acquisition function is numerically optimised,
- and how the quality of the initial design shapes the later trajectory.

In other words, this tutorial is not primarily about introducing a new BO algorithm.

It is about understanding that even within **standard single-loop BoTorch BO**, the behaviour of the optimiser depends strongly on how the modelling problem is set up.

A Bayesian Optimisation workflow is not just:

- a Gaussian Process,
- plus an acquisition function,
- plus a repeated update loop.

It is also shaped by:

- the numerical scale of the inputs,
- the scale and noise level of the outputs,
- the optimisation settings used inside `optimize_acqf`,
- and the information quality of the initial dataset.

These choices affect:

- the stability of GP hyperparameter fitting,
- the geometry of the learned surrogate,
- the quality of the candidate selected at each BO step,
- and how quickly BO can reach strong regions of the search space.

To make these effects visible, this tutorial works directly in a **two-dimensional BO setting**.

That is a deliberate choice.

In one dimension, many practical differences can look subtle.
In two dimensions, they become much easier to see through:

- best-observed-value trajectories,
- sampled-point patterns on the true objective surface,
- and learned hyperparameter behaviour across BO iterations.

So this notebook keeps the BO logic fixed while changing the surrounding modelling conditions one at a time.

---

**This tutorial is designed to shift perspective**
- from *“I know how to run a standard BO loop in BoTorch”*
- to *“I understand which practical modelling choices make that BO loop behave well — or badly — in practice.”*

---

**The emphasis is on developing intuition for**
- why `Normalize` and `Standardize` are usually sensible defaults,
- how noisier observations complicate surrogate fitting and slow optimisation,
- why acquisition optimisation is itself a real numerical subproblem,
- how initial design quality affects sample efficiency,
- and how apparently small modelling choices can alter the full BO trajectory.

---

**Key ideas explored include**
- the effect of **input normalisation** and **output standardisation**,
- the effect of **observation noise** on BO behaviour,
- the role of **`raw_samples`** and **`num_restarts`** in acquisition optimisation,
- the influence of **initial design quality** on optimisation speed,
- and practical interpretation of BO behaviour through 2D performance and sampling-pattern comparisons.

---

This tutorial serves as the practical bridge from:

- **understanding the standard BO loop itself** in Tutorial 3,
- to **understanding how to make that loop work well in realistic BoTorch workflows**.

In other words:

- **Tutorial 1** taught us how to build the surrogate,
- **Tutorial 2** taught us how to act on it through acquisition functions,
- **Tutorial 3** taught us how to run the full sequential BO loop,
- and **Tutorial 4** now asks how practical modelling choices shape the behaviour of that loop in real use.

---

**Recommended prerequisites**
- Completion of **Tutorials 1–3**
- Familiarity with Gaussian Process surrogate modelling in BoTorch
- Familiarity with standard sequential Bayesian Optimisation
- Comfort with one-dimensional and two-dimensional BO visualisations
- Basic awareness of acquisition optimisation through `optimize_acqf`

---

**Author**: Angze Li

**Last updated**: 2026-04-08

**Version**: v1.0

## 🔧 Setup

In [ ]:
import torch
import matplotlib.pyplot as plt

from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.posteriors import GPyTorchPosterior
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.optim import optimize_acqf
from botorch.acquisition.analytic import (
    UpperConfidenceBound,
    LogExpectedImprovement,
    ProbabilityOfImprovement,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")

## 1. Defining the two-dimensional objective and initial BO dataset

To study practical modelling choices in a setting where they matter more visibly, we move from one-dimensional BO to a **two-dimensional scalar-output objective**.

This is useful because many practical effects that can look subtle in 1D become much clearer in 2D. In particular, choices such as:

- input normalisation,
- output standardisation,
- observation noise,
- and acquisition-optimisation settings,

can now affect not just a one-dimensional curve, but the geometry of the sampled-point trajectory across a genuine search surface.

So this cell sets up the full 2D optimisation problem that will be used throughout the tutorial.

---

### What the code does

The function `objective_2d(X)` defines a scalar-valued objective over two input variables:

$$
\mathbf{x} = (x_1, x_2) \in \mathbb{R}^2.
$$

The function combines several components:

- a deeper Gaussian-shaped basin near one region of the plane,
- a second shallower basin elsewhere,
- sinusoidal structure in both input directions,
- and a mild quadratic trend.

Together, these create a surface with:

- multiple attractive regions,
- nontrivial curvature,
- and enough structure that BO does not behave in a trivial or purely local way.

This makes the objective a good testbed for comparing practical BoTorch modelling choices.

---

### Constructing the 2D evaluation grid

The code then defines:

- `x1_grid`
- `x2_grid`

and uses them to construct a full two-dimensional mesh:

```python
X1, X2 = torch.meshgrid(x1_grid, x2_grid, indexing="ij")
```

These grid coordinates are stacked into `X_grid_2d`, which stores the full set of evaluation points in shape `(N, 2)`.

The objective is then evaluated across the entire grid to produce:

```python
Y_grid_2d = objective_2d(X_grid_2d).reshape(100, 100)
```

This gives a dense numerical representation of the true objective surface, which is later used for contour plots.

So this is the 2D analogue of the dense 1D plotting grid used in earlier tutorials.

---

### Defining the BO search bounds

The line

```python
bounds_2d = torch.tensor([[-3.0, -3.0], [3.0, 3.0]], dtype=torch.double)
```

defines the BoTorch search box for the 2D optimisation problem.

This has shape

$$
(2, 2),
$$

where:

- the first row contains the lower bounds for both input dimensions,
- and the second row contains the upper bounds.

So the BO loop will search over

$$
x_1 \in [-3, 3], \qquad x_2 \in [-3, 3].
$$

---

### Defining the initial 2D observations

The tensor `train_X_2d` defines the initial observed input locations.

Each row corresponds to one 2D point:

$$
(x_1, x_2).
$$

The code then evaluates the objective at those locations and adds a small perturbation:

```python
train_Y_2d = objective_2d(train_X_2d) + 0.03 * torch.rand_like(objective_2d(train_X_2d))
```

So the BO loop begins with a **small noisy dataset**, not with full knowledge of the objective surface.

That is exactly the Bayesian Optimisation setting:

- the true surface exists,
- but only a small number of observations are initially available,
- and the surrogate must learn from that partial information.

---

### How to read the figure

The figure shows:

- the full hidden 2D objective as a contour map,
- and the initial observed points as white markers with black edges.

This should be interpreted as the starting information state of the BO loop.

The contour plot is shown only for our understanding.
The BO algorithm itself does **not** know the full surface.
It only begins with the scattered observations shown on top of it.

So the figure makes the central BO problem visually explicit:

> there is a structured objective over a multidimensional search space, but the optimiser only begins with a small and incomplete set of noisy evaluations.

---

### Why this cell matters for the rest of the tutorial

This cell establishes the common 2D setup that will be reused to compare practical modelling decisions.

Because the objective, bounds, and initial observations are fixed here, later differences in BO behaviour can be attributed to changes such as:

- whether transforms are used,
- how noisy the observations are,
- how carefully the acquisition function is optimised,
- or how the initial design is chosen.

So this cell defines the shared baseline for the practical BO comparisons that follow.

---

### Key takeaway

This cell defines the full two-dimensional BO problem used in the tutorial:

- a structured scalar-valued 2D objective,
- a dense grid for visualising the hidden surface,
- a bounded search region,
- and a small noisy initial dataset.

This provides the starting point for examining how practical modelling choices affect Bayesian Optimisation behaviour in a setting where those effects are much easier to see than in one dimension.

In [ ]:
def objective_2d(X):
    x1 = X[..., 0]
    x2 = X[..., 1]
    return (
        -1.15 * torch.exp(-0.5 * ((x1 - 1.1) / 0.45) ** 2 - 0.5 * ((x2 - 1.5) / 0.55) ** 2)
        -0.55 * torch.exp(-0.5 * ((x1 + 1.2) / 0.8) ** 2 - 0.5 * ((x2 + 0.8) / 0.7) ** 2)
        +0.08 * torch.sin(2.2 * x1)
        +0.06 * torch.cos(2.8 * x2)
        +0.03 * (x1**2 + 0.8 * x2**2)
        +0.20
    ).unsqueeze(-1)

x1_grid = torch.linspace(-3.0, 3.0, 100)
x2_grid = torch.linspace(-3.0, 3.0, 100)
X1, X2 = torch.meshgrid(x1_grid, x2_grid, indexing="ij")
X_grid_2d = torch.stack([X1.reshape(-1), X2.reshape(-1)], dim=-1)
Y_grid_2d = objective_2d(X_grid_2d).reshape(100, 100)

bounds_2d = torch.tensor([[-3.0, -3.0], [3.0, 3.0]], dtype=torch.double)

train_X_2d = torch.tensor(
    [
        [-2.2, -2.0],
        [-1.5,  1.0],
        [-0.5, -0.8],
        [ 0.3,  0.5],
        [ 1.0,  2.2],
        [ 1.8,  2.0],
    ],
    dtype=torch.double,
)
train_Y_2d = objective_2d(train_X_2d) + 0.03 * torch.rand_like(objective_2d(train_X_2d))

fig, ax = plt.subplots(figsize=(7, 6))
cont = ax.contourf(X1.numpy(), X2.numpy(), Y_grid_2d.numpy(), levels=30)
ax.scatter(train_X_2d[:, 0].numpy(), train_X_2d[:, 1].numpy(), s=55, color="white", edgecolors="black", zorder=3)
ax.set_title("Initial data for the 2D BO loop", fontsize=18, fontweight="bold")
ax.set_xlabel(r"$x_1$", fontsize=22, fontweight="bold")
ax.set_ylabel(r"$x_2$", fontsize=22, fontweight="bold")
style_ax(ax)
plt.colorbar(cont, ax=ax)
plt.show()

## 2. Defining a reusable BO loop for practical comparisons

Before comparing practical modelling choices, it is helpful to package the standard 2D Bayesian Optimisation workflow into a **single reusable function**.

That is the role of `run_bo_loop_practical(...)`.

This function implements the same core BO cycle used in the previous tutorial:

1. fit a Gaussian Process surrogate to the current data
2. build a `LogExpectedImprovement` acquisition function
3. optimise that acquisition function over the search domain
4. evaluate the objective at the selected point
5. update the dataset
6. and repeat

What is new here is that the function has been written in a more **configurable** and more **computationally efficient** way, so that it can be reused for controlled experiments throughout this notebook.

---

### What the function does

The function takes:

- an initial observed dataset: `train_X_init`, `train_Y_init`
- the true objective function `objective_fn`
- the BO search bounds `bounds`
- the number of BO updates `n_steps`

and then performs a standard sequential BO loop.

At each iteration, the code:

- converts the minimisation problem into a maximisation-compatible form with
  `train_Y_bo = -train_Y`
- fits a `SingleTaskGP`
- builds a `LogExpectedImprovement` acquisition function
- optimises it with `optimize_acqf`
- stores the current BO state
- and, unless the loop is finished, appends the newly acquired observation to the dataset

So this function is the practical workhorse that drives all later comparisons.

---

## i) Modifications that allow us to test different practical variables

This function has been made configurable so that different modelling choices can be changed **without rewriting the BO loop itself**.

That is the main reason this notebook can compare multiple practical BO settings cleanly.

### 1. Optional input normalisation

The argument

```python
use_input_transform=True
```

controls whether the GP uses

```python
Normalize(d=train_X.shape[-1])
```

internally.

This allows us to compare:

- BO **with** input normalisation
- versus BO **without** input normalisation

using exactly the same loop.

---

### 2. Optional output standardisation

The argument

```python
use_outcome_transform=True
```

controls whether the GP uses

```python
Standardize(m=1)
```

internally.

This allows us to compare:

- BO **with** output standardisation
- versus BO **without** output standardisation

again without changing any other part of the BO workflow.

Together, these two switches make it possible to study the effect of transforms in a controlled way.

---

### 3. Configurable acquisition-optimisation settings

The arguments

- `num_restarts`
- `raw_samples`

are exposed directly in the function signature.

This allows us to test different acquisition-optimisation regimes, for example:

- weaker acquisition optimisation with fewer restarts and fewer raw initial samples
- stronger acquisition optimisation with more restarts and more raw initial samples

So the function can be reused not only for transform comparisons, but also for studying how carefully the acquisition function itself is numerically optimised.

---

### 4. Flexible evaluation grid for optional posterior storage

The argument

```python
X_eval=None
```

allows us to pass in an external evaluation grid when needed.

This makes the function flexible enough to support two different usage modes:

- a lightweight BO loop used only for performance comparison
- or a richer BO loop that also stores posterior quantities for plotting

So the function is no longer tied to one particular plotting grid or one fixed analysis workflow.

---

### 5. Optional posterior storage

The argument

```python
store_posterior=False
```

controls whether the code computes and stores:

- posterior mean `mu`
- posterior standard deviation `sigma`

at each BO iteration.

This is useful because some notebook sections need full posterior grids for contour plots, while others only need:

- best observed values
- final sample locations
- learned hyperparameters

So this flag makes the function suitable for multiple experimental purposes.

---

### 6. Optional reshaping through `grid_shape`

The argument

```python
grid_shape=None
```

allows stored posterior quantities to be reshaped to the geometry of a 2D plotting grid when needed.

That means the same function can store posterior values either:

- as flat tensors
- or in a contour-plot-friendly 2D array shape

depending on the downstream visualisation task.

This makes the function more reusable and less tied to any one plotting format.

---

## ii) Modifications that accelerate execution time

A second major improvement is that the function has been rewritten to avoid unnecessary heavy computations during repeated BO runs.

This matters especially in 2D, where evaluating GP posteriors on a dense grid can become expensive.

### 1. Posterior computation is now optional

This is the most important speed improvement.

In earlier versions, the code always computed:

- `posterior = gp.posterior(...)`
- posterior mean
- posterior standard deviation

at every BO step, even when these quantities were not needed for the final comparison.

That is expensive, especially in 2D.

Now, posterior evaluation only happens when both of the following are true:

- `store_posterior=True`
- and `X_eval is not None`

So for many comparison sections — such as best-value curves or final sample-pattern plots — the BO loop no longer wastes time computing full posterior grids that are never used.

This substantially reduces runtime.

---

### 2. The evaluation grid is no longer mandatory

Because `X_eval` is optional, the BO loop can now be run without any dense grid at all when the goal is simply to compare optimisation outcomes.

That avoids repeated GP posterior evaluation across large 2D grids.

This is particularly important because dense 2D contour grids can easily contain thousands or tens of thousands of points, and repeatedly evaluating a GP posterior on such grids can dominate execution time.

---

### 3. Posterior reshaping is only done when needed

The reshaping logic via `grid_shape` is only invoked when posterior storage is actually requested.

So even the extra tensor manipulations associated with contour plotting are skipped in lightweight runs.

---

### 4. The BO loop remains reusable without duplicating expensive plotting logic

By separating:

- BO optimisation logic
- from posterior-grid plotting logic

the function avoids mixing heavy visualisation preparation into every experimental run.

This is not only cleaner conceptually — it is also faster computationally.

---

## Other useful outputs stored in `history`

At each BO iteration, the function stores a dictionary containing:

- the current step index
- the current dataset
- optional posterior quantities
- the selected candidate point
- the acquisition value at that candidate
- the learned kernel lengthscales
- and the learned likelihood noise

This is helpful because it allows later notebook sections to analyse BO behaviour from different perspectives without rerunning the loop unnecessarily.

For example, we can later inspect:

- best observed value over time
- final sampling patterns
- learned hyperparameter trajectories
- or posterior contour plots

from the same stored BO history.

---

## Why this function is important for the tutorial

This function is the core computational engine for the rest of the notebook.

It allows the tutorial to keep the **same BO structure fixed** while changing only one modelling choice at a time.

That makes later comparisons much more interpretable, because the observed differences can be attributed to:

- the use or omission of transforms,
- the noise level,
- the acquisition-optimisation settings,
- or the initial design,

rather than to differences in the BO loop itself.

So this function is what makes the whole practical-comparison structure of the tutorial possible.

---

### Key takeaway

`run_bo_loop_practical(...)` is a reusable 2D Bayesian Optimisation loop designed specifically for practical experimentation.

It has been modified in two important ways:

- first, to allow controlled testing of different BO choices through configurable transforms, acquisition-optimisation settings, and optional posterior storage
- and second, to accelerate execution by avoiding expensive posterior-grid computations unless they are explicitly needed for plotting

This makes it a practical and flexible base for all of the BO comparisons that follow.

In [ ]:
def run_bo_loop_practical(
    train_X_init,
    train_Y_init,
    objective_fn,
    bounds,
    n_steps=8,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=10,
    raw_samples=50,
    X_eval=None,
    store_posterior=False,
    grid_shape=None,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()

    history = []

    for step in range(n_steps + 1):
        train_Y_bo = -train_Y
        y_best_bo = train_Y_bo.max().item()

        input_tf = Normalize(d=train_X.shape[-1]) if use_input_transform else None
        outcome_tf = Standardize(m=1) if use_outcome_transform else None

        gp = SingleTaskGP(
            train_X=train_X,
            train_Y=train_Y_bo,
            input_transform=input_tf,
            outcome_transform=outcome_tf,
        )
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)
        gp.eval()

        if store_posterior and X_eval is not None:
            posterior = gp.posterior(X_eval)
            mu_raw = -posterior.mean.detach()
            sigma_raw = torch.sqrt(posterior.variance.detach().clamp_min(1e-12))

            if grid_shape is not None:
                mu = mu_raw.reshape(grid_shape)
                sigma = sigma_raw.reshape(grid_shape)
            else:
                mu = mu_raw
                sigma = sigma_raw
        else:
            mu = None
            sigma = None

        log_ei = LogExpectedImprovement(model=gp, best_f=y_best_bo)
        candidate, acq_value = optimize_acqf(
            acq_function=log_ei,
            bounds=bounds,
            q=1,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )

        lengthscale = gp.covar_module.lengthscale.detach().cpu().reshape(-1).numpy()
        noise = gp.likelihood.noise.detach().cpu().item()

        history.append({
            "step": step,
            "train_X": train_X.clone(),
            "train_Y": train_Y.clone(),
            "mu": None if mu is None else mu.clone(),
            "sigma": None if sigma is None else sigma.clone(),
            "candidate": candidate.clone(),
            "acq_value": acq_value.clone(),
            "lengthscale": lengthscale.copy(),
            "noise": noise,
        })

        if step < n_steps:
            x_new = candidate
            y_new = objective_fn(x_new)
            train_X = torch.cat([train_X, x_new], dim=0)
            train_Y = torch.cat([train_Y, y_new], dim=0)

    return history

## 3. Running two BO loops: with transforms and without transforms



We now use the same reusable 2D BO loop to compare two modelling choices:

- **with BoTorch transforms**
- **without BoTorch transforms**

This is one of the most important practical comparisons in the notebook, because it isolates the effect of **input normalisation** and **output standardisation** on otherwise identical BO runs.

---

### What we are comparing

The two calls differ only in these two arguments:

- `use_input_transform`
- `use_outcome_transform`

In the first run, we use:

- `use_input_transform=True`
- `use_outcome_transform=True`

so the GP is fit with:

- `Normalize(d=2)` on the inputs
- `Standardize(m=1)` on the outputs

In the second run, we use:

- `use_input_transform=False`
- `use_outcome_transform=False`

so the GP is fit directly on the raw input coordinates and raw observed objective values.

Everything else is kept fixed:

- the same 2D objective
- the same initial observed dataset
- the same BO bounds
- the same number of BO steps
- and the same acquisition-optimisation settings

So any difference we later observe in BO behaviour can be attributed primarily to the use or omission of these standard transforms.

---

### Why this is a meaningful practical comparison

In BoTorch, using input and output transforms is usually a good default.

That is because GP fitting is often more stable and better conditioned when:

- inputs are mapped to a common numerical scale
- and outputs are centred and rescaled to a more standard range

When these transforms are omitted, the surrogate must fit directly on:

- raw input coordinates
- and raw output values

which can make hyperparameter fitting less well behaved.

This can affect:

- the learned lengthscales
- the learned noise
- the shape of the posterior
- and ultimately the BO trajectory itself

So this section is asking a very practical question:

> if we keep the BO loop fixed, how much does it matter whether we apply the standard BoTorch transforms or not?

---

### Why `store_posterior=True` is used here

In both runs, we also set:

- `store_posterior=True`
- `grid_shape=X1.shape`

This means that, at each BO iteration, the function stores the GP posterior mean and uncertainty evaluated on the 2D plotting grid `X_grid_2d`.

That is useful here because this comparison is not only about the final best observed value.
We also want to inspect how the learned surrogate surface itself differs between:

- the transformed model
- and the untransformed model

So for this section, it is worth paying the additional computational cost of storing posterior grids.

---

### Why omitting the input transform produces an input-data warning

When `use_input_transform=False`, the GP receives the raw 2D coordinates directly.

But BoTorch expects standard GP inputs to be approximately contained in the **unit cube**:

$$
[0,1]^d.
$$

In this tutorial, however, the raw 2D search space is

$$
x_1, x_2 \in [-3, 3].
$$

So the untransformed data are clearly **not** in the unit cube.

That is why BoTorch emits an `InputDataWarning` saying that the input features are not min-max scaled.

This warning is not saying the code is invalid.
It is simply telling us that the model is being fit in a form that is **not the recommended default**.

In other words:

- the code still runs
- but the GP is being asked to learn from inputs on a less convenient numerical scale

That can make the fit less stable and less interpretable.

---

### Why omitting the outcome transform also produces a warning

Similarly, when `use_outcome_transform=False`, the observed outputs are no longer standardised internally.

BoTorch then checks whether the observed outputs are already approximately:

- zero mean
- and unit variance

In this case they are not, so BoTorch warns that the outcome observations are not standardised.

Again, this is not a fatal error.
It is simply a signal that the model is being fit without the usual numerical preprocessing that often improves GP behaviour.

So the warnings in the “without transforms” run are actually part of the lesson:

> they reflect that we are deliberately disabling the standard BoTorch preprocessing in order to see what difference it makes.

---

### Why this comparison is especially useful in 2D

In one dimension, the effect of transforms can sometimes look subtle.

In two dimensions, the impact is often much easier to see, because differences in the GP fit can show up not only in the best-value trajectory, but also in:

- the geometry of the posterior surface
- the final sample locations
- and the learned lengthscales in each input direction

So this 2D setting makes the practical consequences of using or omitting transforms much more visible.

---

### Key takeaway

This cell runs two otherwise identical 2D BO loops in order to compare:

- a GP fit with standard BoTorch transforms
- and a GP fit without them

The purpose is to isolate the practical effect of **input normalisation** and **output standardisation**.

The `InputDataWarning` in the untransformed case appears because the raw input coordinates lie in `[-3, 3]^2`, not in the unit cube `[0,1]^2`, which is the scale BoTorch typically recommends for GP modelling.

In [ ]:
history_with_tf = run_bo_loop_practical(
    train_X_init=train_X_2d,
    train_Y_init=train_Y_2d,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    X_eval=X_grid_2d,
    n_steps=50,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=10,
    raw_samples=50,
    store_posterior=True,
    grid_shape=X1.shape,
)

history_without_tf = run_bo_loop_practical(
    train_X_init=train_X_2d,
    train_Y_init=train_Y_2d,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    X_eval=X_grid_2d,
    n_steps=50,
    use_input_transform=False,
    use_outcome_transform=False,
    num_restarts=10,
    raw_samples=50,
    store_posterior=True,
    grid_shape=X1.shape,
)

## 4. Comparing BO performance and final sampling patterns with and without transforms

Now that the two BO runs have been completed, we can compare their behaviour from two complementary perspectives:

- how the **best observed value** evolves over BO iterations
- and where the optimiser ultimately chooses to **place evaluations** on the true 2D objective surface

This gives a practical view of how input normalisation and output standardisation affect Bayesian Optimisation in a multidimensional setting.

---

### What the code does

The first two lines extract, from each stored BO state, the current best observed objective value:

```python
best_with_tf = [float(torch.min(h["train_Y"])) for h in history_with_tf]
best_without_tf = [float(torch.min(h["train_Y"])) for h in history_without_tf]
```

Because this is a minimisation problem, lower values are better.
So these two sequences track how quickly each BO run improves the best solution it has found so far.

The figure then creates two side-by-side panels.

#### Left panel
The left panel plots:

- the best observed value over BO iterations for the run **with transforms**
- and the best observed value over BO iterations for the run **without transforms**

This gives a direct optimisation-performance comparison.

#### Right panel
The right panel uses the **true objective surface** `Y_grid_2d` as a contour background, and overlays the final sampled locations from the two BO runs:

- square markers for the run **without transforms**
- circular markers for the run **with transforms**

So this panel shows how the two modelling choices lead to different final spatial allocations of the BO budget.

---

### How to read the left panel

The left panel tracks

```python
min(y_observed)
```

as a function of BO iteration.

This should be interpreted as the practical optimisation outcome:

- if one curve drops earlier or lower, that BO run is finding better objective values sooner
- if one curve plateaus for longer, that run is spending more time without improving the current best value

So the left panel answers the practical question:

> **Does using transforms actually help the BO loop find better points more efficiently?**

This is the most direct performance comparison in the section.

---

### How to read the right panel

The right panel should be interpreted differently.

Here, the background contour is the **true hidden objective**, while the markers show the final set of points sampled by each BO run.

This answers a different question:

> **Did the two BO runs end up exploring and exploiting the search space in the same way, or did they allocate evaluations differently?**

Because Bayesian Optimisation is sequential, differences in early surrogate fitting can lead to different candidate choices, which then lead to different datasets, which then further change the surrogate.
So the BO trajectory is path dependent.

That is why this panel is so useful:
it makes the accumulated effect of those sequential modelling decisions spatially visible.

---

### Why this figure is enough for the main comparison

This figure is doing the core practical work of the section.

It shows:

- whether the two BO runs differ in optimisation performance
- and whether they differ in where they ultimately sampled

That is enough to establish that the choice of transforms is not just a cosmetic modelling detail.
It can change the BO process in an observable way.

A more diagnostic posterior-surface comparison could also be plotted, but it is not strictly necessary here, because once the two runs follow different sampling trajectories, they will naturally end up fitting different surrogates.

So this figure keeps the comparison focused on the most practically important outcomes.

---

### Why different sampling trajectories imply different fitted surrogates

It is worth making one point explicit.

A Gaussian Process surrogate is fitted to the currently observed dataset.
So if two BO runs sample different points, then they end up with different observed datasets.

Once that happens, the fitted surrogates must also differ, because they are being conditioned on different information.

So:

- different transforms can lead to different BO decisions
- different BO decisions lead to different sampled datasets
- and different sampled datasets imply different fitted surrogates

That is why the final sampling-pattern comparison is already highly informative, even without separately plotting the final posterior surfaces.

---

### Why this comparison is especially useful in 2D

In one dimension, the effect of transforms may show up only as relatively small shifts in a curve.

In two dimensions, those same effects can become much more visible because they influence:

- the route the optimiser takes through the search space
- the regions where it concentrates
- and whether it explores broadly or commits early to a particular basin

So this figure makes the practical consequences of preprocessing decisions much easier to see.

---

### Key takeaway

This figure compares two BO runs that differ only in whether standard BoTorch transforms are used.

The left panel shows the effect on **optimisation performance** through the best observed value over time.
The right panel shows the effect on **sampling behaviour** through the final point locations on the true 2D objective.

Together, these plots show that using or omitting transforms can change not only the numerical GP fit, but also the full sequential BO trajectory.

In [ ]:
best_with_tf = [float(torch.min(h["train_Y"])) for h in history_with_tf]
best_without_tf = [float(torch.min(h["train_Y"])) for h in history_without_tf]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

axes[0].plot(range(len(best_with_tf)), best_with_tf, "-o", linewidth=2.5, label="With transforms")
axes[0].plot(range(len(best_without_tf)), best_without_tf, "-o", linewidth=2.5, label="Without transforms")
axes[0].set_title("Effect of transforms on 2D BO performance", fontsize=18, fontweight="bold")
axes[0].set_xlabel("BO iteration", fontsize=22, fontweight="bold")
axes[0].set_ylabel("Best observed value", fontsize=22, fontweight="bold")
axes[0].legend(prop={"size": 10, "weight": "bold"})
style_ax(axes[0])

h_with = history_with_tf[-1]
h_without = history_without_tf[-1]
cont = axes[1].contourf(X1.numpy(), X2.numpy(), Y_grid_2d.numpy(), levels=30)
axes[1].scatter(
    h_without["train_X"][:, 0].numpy(),
    h_without["train_X"][:, 1].numpy(),
    s=60,
    marker="s",
    facecolors="none",
    edgecolors="black",
    linewidths=1.2,
    zorder=3,
    label="Without transforms",
)
axes[1].scatter(
    h_with["train_X"][:, 0].numpy(),
    h_with["train_X"][:, 1].numpy(),
    s=55,
    marker="o",
    color="white",
    edgecolors="black",
    linewidths=1.0,
    zorder=4,
    label="With transforms",
)
axes[1].set_title("Final sampling pattern: \n with vs without transforms", fontsize=18, fontweight="bold")
axes[1].set_xlabel(r"$x_1$", fontsize=22, fontweight="bold")
axes[1].set_ylabel(r"$x_2$", fontsize=22, fontweight="bold")
axes[1].legend(prop={"size": 9, "weight": "bold"}, loc="upper left")
style_ax(axes[1])
plt.colorbar(cont, ax=axes[1])

plt.tight_layout()
plt.show()

## 5. Comparing learned lengthscales and noise with and without transforms

The previous figure compared the two BO runs from the perspective of:

- optimisation performance
- and final sampling behaviour

This figure now looks one layer deeper and asks:

> **How do the underlying GP hyperparameters evolve across BO iterations under the two modelling choices?**

In particular, we compare:

- the learned **lengthscales** in the two input directions
- and the learned **likelihood noise**

This is useful because transforms do not only change the sampled trajectory externally.
They also change how the GP internally interprets distances, smoothness, and observational uncertainty.

---

### What the code does

The code first extracts from the stored BO history:

- `lengthscale`
- and `noise`

for each BO step in the two runs:

- `history_with_tf`
- `history_without_tf`

Because this is a 2D problem, each learned lengthscale has two components:

- one for $x_1$ direction
- and one for $x_2$ direction

So the code separates them into:

- `length_with_tf_x1`
- `length_with_tf_x2`
- `length_without_tf_x1`
- `length_without_tf_x2`

The figure then has two panels:

#### Left panel
The left panel plots the learned GP lengthscales over BO iterations.

#### Right panel
The right panel plots the learned GP likelihood noise over BO iterations.

So this figure is not about the true objective directly.
It is about how the GP model is adapting its own internal hyperparameters as the BO loop progresses.

---

### What the learned lengthscale means

In a GP kernel, the lengthscale controls how quickly the model expects the function to vary as we move through the input space.

Roughly speaking:

- **small lengthscale**
  means the model allows the function to change rapidly over short distances

- **large lengthscale**
  means the model expects smoother, more slowly varying behaviour

Because this is a 2D model, the GP can learn a different lengthscale in each input direction.
That means it can decide that the function varies more rapidly along one axis than along the other.

So the left panel tells us how the GP is learning the effective geometry of the objective surface.

---

### Interpreting the left panel: lengthscale behaviour

The most striking feature is that the run **with transforms** learns much more stable and much smaller lengthscales, while the run **without transforms** initially shows very large and erratic values.

This is exactly the kind of behaviour we would expect.

#### With transforms
When the inputs are normalised to the unit cube and the outputs are standardised, the GP hyperparameter optimisation problem is much better conditioned.

So the learned lengthscales:

- remain on a more interpretable numerical scale
- vary more smoothly over BO iterations
- and stabilise relatively quickly

This suggests that the surrogate is fitting the data in a numerically coherent way.

#### Without transforms
Without normalisation and standardisation, the GP is fitting directly on raw coordinates in $[-3,3]^2$ and raw observed objective values.

That makes hyperparameter fitting less well conditioned.

As a result, the learned lengthscales:

- become much larger
- fluctuate more strongly in the early BO steps
- and differ much more dramatically across the two input directions

This indicates that the model is having a harder time settling on a stable notion of distance and smoothness in the raw input space.

So the left panel gives a strong internal explanation for why the transformed and untransformed BO runs can behave differently.

---

### Why the no-transform lengthscales are so much larger

This is an important point.

The absolute numerical value of the lengthscale depends on the scale of the input coordinates.

With transforms:

- inputs are mapped roughly into $[0,1]^2$

Without transforms:

- inputs stay in $[-3,3]^2$

So even before worrying about optimisation stability, the “natural” scale of the lengthscale parameters is already very different.

That is why the untransformed run can easily produce much larger learned lengthscales numerically.

But the key issue is not just that they are larger.
It is that they are also:

- more erratic
- less stable early on
- and less easy to interpret

That is the practical problem the transforms are helping to avoid.

---

### What the learned noise means

The GP likelihood noise is the model’s estimate of how noisy the observed outputs are.

Roughly speaking:

- larger noise means the GP is attributing more variation in the data to observation noise
- smaller noise means the GP is treating the data as more directly informative about the latent function

So the right panel tells us how much observational uncertainty the GP thinks it needs in order to explain the data at each BO step.

---

### Interpreting the right panel: noise behaviour

The right panel again shows a clear contrast between the transformed and untransformed runs.

#### With transforms
The transformed run learns a noise level that:

- is smaller
- decreases relatively smoothly
- and stabilises in a controlled way

This suggests that the GP fit is numerically well behaved and that the model is not having to overcompensate with an artificially inflated noise term.

#### Without transforms
The untransformed run shows a much more erratic and initially much larger noise estimate.

This is also consistent with what we would expect.

When the GP is fit on raw, unscaled inputs and outputs, some modelling mismatch that should ideally be absorbed by better-scaled kernel hyperparameters can instead get partially pushed into the learned noise term.

So the model without transforms may temporarily interpret instability or mismatch as “noise,” even when the underlying issue is really poor conditioning of the hyperparameter fit.

That is why the orange curve starts higher and fluctuates more strongly before eventually decreasing.

---

### Why both noise curves eventually become small

A useful observation is that both runs eventually push the learned noise toward very small values.

That is not surprising here, because:

- the synthetic objective itself is deterministic
- and the observation perturbation added at the start is relatively small

So as the dataset grows, both models can eventually fit the surface more confidently.

But the important difference is how they get there:

- the transformed run gets there more smoothly
- the untransformed run takes a noisier and less stable path

That difference is pedagogically important.

---

### What this figure adds beyond the earlier performance plot

The earlier best-value and sampling-pattern figure already showed that the transformed and untransformed BO runs behave differently.

This figure now explains *why* from the GP’s internal perspective.

It shows that using or omitting transforms changes:

- the effective notion of smoothness learned by the kernel
- and the amount of observational noise the model thinks it needs

So the consequence is not merely cosmetic.
The transforms change the internal statistical model that drives the BO decisions.

---

### Why this supports the practical recommendation to use transforms

This figure provides a strong practical justification for the standard BoTorch recommendation:

- normalise inputs
- standardise outputs

The transformed run produces hyperparameter trajectories that are:

- numerically more stable
- easier to interpret
- and less erratic in the early stages of BO

That does not mean transforms magically guarantee perfect BO behaviour.
But they do make the surrogate fitting problem much better behaved.

And since BO decisions are driven by the surrogate, that matters a lot.

---

### Key takeaway

This figure compares how the GP hyperparameters evolve across BO iterations with and without transforms.

The transformed run learns:

- more stable and interpretable lengthscales
- and a smoother, smaller noise trajectory

The untransformed run shows:

- much larger and more erratic early lengthscales
- and a noisier likelihood fit

So this figure makes the internal modelling effect of preprocessing explicit:
using normalisation and standardisation does not just change the numerical scale of the data — it changes how stably and sensibly the GP learns the geometry and uncertainty structure of the problem.

In [ ]:
length_with_tf = [h["lengthscale"] for h in history_with_tf]
length_without_tf = [h["lengthscale"] for h in history_without_tf]

noise_with_tf = [h["noise"] for h in history_with_tf]
noise_without_tf = [h["noise"] for h in history_without_tf]

length_with_tf_x1 = [vals[0] for vals in length_with_tf]
length_with_tf_x2 = [vals[1] for vals in length_with_tf]
length_without_tf_x1 = [vals[0] for vals in length_without_tf]
length_without_tf_x2 = [vals[1] for vals in length_without_tf]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(len(length_with_tf_x1)), length_with_tf_x1, "-o", linewidth=2.0, label="With transforms: $x_1$")
axes[0].plot(range(len(length_with_tf_x2)), length_with_tf_x2, "-o", linewidth=2.0, label="With transforms: $x_2$")
axes[0].plot(range(len(length_without_tf_x1)), length_without_tf_x1, "--s", linewidth=2.0, label="Without transforms: $x_1$")
axes[0].plot(range(len(length_without_tf_x2)), length_without_tf_x2, "--s", linewidth=2.0, label="Without transforms: $x_2$")
axes[0].set_title("Learned 2D lengthscales over BO iterations", fontsize=16, fontweight="bold")
axes[0].set_xlabel("BO iteration", fontsize=18, fontweight="bold")
axes[0].set_ylabel("Lengthscale", fontsize=18, fontweight="bold")
axes[0].legend(prop={"size": 8, "weight": "bold"})
style_ax(axes[0])

axes[1].plot(range(len(noise_with_tf)), noise_with_tf, "-o", linewidth=2.0, label="With transforms")
axes[1].plot(range(len(noise_without_tf)), noise_without_tf, "-o", linewidth=2.0, label="Without transforms")
axes[1].set_title("Learned noise over BO iterations", fontsize=16, fontweight="bold")
axes[1].set_xlabel("BO iteration", fontsize=18, fontweight="bold")
axes[1].set_ylabel("Noise", fontsize=18, fontweight="bold")
axes[1].legend(prop={"size": 9, "weight": "bold"})
style_ax(axes[1])

plt.tight_layout()
plt.show()

## 6. Running two BO loops under different observation-noise levels

We now compare two BO runs that differ only in the **noise level of the observed data**.

This is an important practical experiment because, in real optimisation problems, the objective is often not observed perfectly.
Measurements may be affected by:

- instrumental noise,
- stochastic simulation variability,
- imperfect experimental control,
- or other sources of uncertainty.

So this section asks a very practical question:

> **How does Bayesian Optimisation behaviour change when the same objective is observed with different amounts of noise?**

---

### What we are comparing

The first two lines construct two versions of the initial observed dataset:

- a **low-noise** version
- and a **high-noise** version

using the same underlying 2D objective:

```python
train_Y_low_noise = objective_2d(train_X_2d) + 0.01 * torch.rand_like(objective_2d(train_X_2d))
train_Y_high_noise = objective_2d(train_X_2d) + 0.10 * torch.rand_like(objective_2d(train_X_2d))
```

So the only difference here is the magnitude of the added perturbation:

- `0.01` for the low-noise case
- `0.10` for the high-noise case

The input locations `train_X_2d` are identical in both runs.
That means the comparison isolates the effect of **observation quality**, not input placement.

---

### Why this is a meaningful BO comparison

Bayesian Optimisation relies on a surrogate model that must infer the latent objective from noisy observed values.

If the observations are only mildly noisy, the surrogate can treat them as fairly reliable evidence about the underlying function.

If the observations are more strongly noisy, the surrogate must be more cautious.
This can affect:

- the learned likelihood noise
- the fitted posterior surface
- the acquisition function
- and therefore the full sequential BO trajectory

So the goal here is to see how much BO behaviour changes when the same optimisation problem is presented with cleaner versus noisier data.

---

### What the code does

The code runs the same BO loop twice:

- once with `train_Y_low_noise`
- once with `train_Y_high_noise`

All other settings are kept fixed:

- the same 2D objective
- the same initial input locations
- the same search bounds
- the same number of BO steps
- the same use of input normalisation and output standardisation
- and the same acquisition-optimisation settings

So, once again, any later difference in BO behaviour can be attributed mainly to the change in observation noise.

---

### Why transforms are still kept on here

Both runs use:

- `use_input_transform=True`
- `use_outcome_transform=True`

This is important because the purpose of this section is **not** to test transforms again.
It is to isolate the effect of observation noise.

By keeping the standard BoTorch preprocessing turned on, we avoid mixing the noise experiment with the transform experiment.
That makes the comparison cleaner and easier to interpret.

---

### Why `store_posterior=True` is used again

Both runs also set:

- `store_posterior=True`
- `grid_shape=X1.shape`

This means the BO history stores posterior-grid information as the loop evolves.

Even though the immediate comparison may focus mainly on:

- best observed values
- and final sample locations

it is still useful to retain posterior information for later diagnostic interpretation if needed.

So in this section, we again choose the more information-rich BO run rather than the lightest possible one.

---

### What we expect before seeing the results

Before plotting anything, it is useful to anticipate the qualitative effect of higher noise.

Compared with the low-noise run, the high-noise run may:

- fit a more uncertain surrogate
- learn a larger likelihood noise
- improve the best observed value more slowly
- or allocate evaluations differently because the model has less confidence in what it has seen

So the comparison is not only about whether the final answer is better or worse.
It is also about whether the BO process becomes less stable or less decisive when the data are noisier.

---

### Why this matters in practice

This section is especially relevant because many real BO applications are noisy by default.

For example:

- laboratory measurements are rarely exact
- optimisation over simulations may involve stochastic outputs
- repeated evaluations of the same setting may not produce identical values

So understanding how BO responds to different noise levels is essential for using it sensibly in practice.

A workflow that works beautifully on nearly noiseless synthetic data may behave quite differently once realistic observation noise is introduced.

---

### Key takeaway

This cell runs two otherwise identical 2D BO loops in order to compare:

- a **low-noise observation regime**
- and a **high-noise observation regime**

The only thing that changes is the magnitude of the noise added to the initial observed outputs.

This allows the rest of the section to isolate how observation noise affects:

- the surrogate fit,
- the BO trajectory,
- and the practical optimisation behaviour of the loop.

In [ ]:
train_Y_low_noise = objective_2d(train_X_2d) + 0.01 * torch.rand_like(objective_2d(train_X_2d))
train_Y_high_noise = objective_2d(train_X_2d) + 0.10 * torch.rand_like(objective_2d(train_X_2d))

history_low_noise = run_bo_loop_practical(
    train_X_init=train_X_2d,
    train_Y_init=train_Y_low_noise,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    X_eval=X_grid_2d,
    n_steps=50,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=10,
    raw_samples=50,
    store_posterior=True,
    grid_shape=X1.shape,
)

history_high_noise = run_bo_loop_practical(
    train_X_init=train_X_2d,
    train_Y_init=train_Y_high_noise,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    X_eval=X_grid_2d,
    n_steps=50,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=10,
    raw_samples=50,
    store_posterior=True,
    grid_shape=X1.shape,
)

## 7. Comparing BO performance and final sampling patterns under different noise levels

Now that the two BO runs have been completed, we can compare the practical effect of **observation noise** from two perspectives:

- how the **best observed value** evolves over BO iterations
- and where the optimiser ultimately chooses to **place evaluations** on the true 2D objective surface

This gives a direct view of how noise changes the behaviour of the full Bayesian Optimisation loop.

---

### What the code does

The first two lines extract, from each stored BO state, the current best observed objective value:

```python
best_low_noise = [float(torch.min(h["train_Y"])) for h in history_low_noise]
best_high_noise = [float(torch.min(h["train_Y"])) for h in history_high_noise]
```

Because this is a minimisation problem, lower values are better.
So these two sequences track how quickly each BO run improves the best solution it has found so far.

The figure then creates two side-by-side panels.

#### Left panel
The left panel plots:

- the best observed value over BO iterations for the **low-noise** run
- and the best observed value over BO iterations for the **high-noise** run

This gives a direct optimisation-performance comparison.

#### Right panel
The right panel uses the **true objective surface** `Y_grid_2d` as a contour background, and overlays the final sampled locations from the two BO runs:

- circular white markers for the **low-noise** run
- hollow square markers for the **high-noise** run

So this panel shows how the two noise levels lead to different final spatial allocations of the BO budget.

---

### How to read the left panel

The left panel tracks

```python
min(y_observed)
```

as a function of BO iteration.

This should be interpreted as the practical optimisation outcome:

- if one curve drops earlier or lower, that BO run is finding better objective values sooner
- if one curve plateaus for longer, that run is spending more time without improving the current best value

In the attached result, the contrast is quite striking:

- the **low-noise** run eventually discovers a much better basin and reaches a substantially lower best value
- the **high-noise** run remains trapped for much longer around a shallower region and does not achieve the same late-stage improvement

So the main practical message from the left panel is:

> **higher observation noise can make BO much slower to discover the truly best region of the search space.**

---

### How to read the right panel

The right panel should be interpreted differently.

Here, the background contour is the **true hidden objective**, while the markers show the final set of points sampled by each BO run.

This answers a different question:

> **Did the two BO runs allocate their evaluations in the same way, or did noise change the sampling trajectory?**

In the attached result, the final sampling patterns are clearly different.

The **low-noise** run places more evaluations in and around the lower basin near the bottom-left region, and eventually succeeds in pushing into the deeper minimum there.

By contrast, the **high-noise** run spreads its evaluations differently and appears to spend more effort without decisively committing to the truly best basin.

So the right panel makes the path-dependent effect of noisy data visually explicit.

---

### What the attached result is telling us

This particular result is pedagogically very useful.

It shows that even when:

- the underlying objective is the same
- the BO loop is the same
- the bounds are the same
- and the acquisition settings are the same

changing only the amount of observation noise can produce a very different optimisation outcome.

In this run:

- the **low-noise** BO loop eventually finds the deeper low-value region
- while the **high-noise** BO loop remains effectively stuck near a shallower regime

So the difference is not just cosmetic.
It changes both:

- the best objective value reached
- and the part of the search space the BO loop ends up focusing on

---

### Why this happens

Bayesian Optimisation acts on a surrogate model, not directly on the true objective.

When the data are noisier:

- the GP has less reliable evidence about the latent function
- the posterior becomes less decisive
- and the acquisition function may struggle to distinguish genuinely promising regions from noisy observations

That makes the BO trajectory more fragile.

In other words:

- cleaner observations help the surrogate form a sharper belief about where the good regions are
- noisier observations make the surrogate more cautious and can delay or prevent discovery of the global basin

That is exactly the behaviour illustrated by the attached figure.

---

### Why different noise levels imply different fitted surrogates

As in the transform comparison, it is worth making one point explicit.

The GP surrogate is fitted to the observed dataset.

So if the observed values differ because of noise, then even with the same input locations, the fitted surrogate will differ.
Once the BO loop begins sampling different new points, the datasets diverge even further.

So:

- different noise levels lead to different observed targets
- different observed targets lead to different surrogate fits
- and different surrogate fits lead to different BO decisions

That is why differences in noise can propagate through the entire sequential optimisation process.

---

### Why this comparison is especially useful in 2D

In one dimension, the effect of noise may look like a small perturbation to a curve.

In two dimensions, the same effect can be much more consequential, because it can change:

- which basin the optimiser commits to
- whether the search remains spread out or becomes focused
- and whether BO ever reaches the truly best region within the available evaluation budget

So the 2D setting makes the practical consequences of noisy observations much easier to see.

---

### Key takeaway

This figure compares two BO runs that differ only in the amount of observation noise in the initial data.

The left panel shows the effect on **optimisation performance** through the best observed value over time.
The right panel shows the effect on **sampling behaviour** through the final point locations on the true 2D objective.

In the attached result, the low-noise run clearly outperforms the high-noise run, illustrating an important practical lesson:

> **higher observation noise can substantially slow Bayesian Optimisation and can prevent the BO loop from confidently discovering the truly best basin within a limited evaluation budget.**

In [ ]:
best_low_noise = [float(torch.min(h["train_Y"])) for h in history_low_noise]
best_high_noise = [float(torch.min(h["train_Y"])) for h in history_high_noise]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

axes[0].plot(range(len(best_low_noise)), best_low_noise, "-o", linewidth=2.5, label="Low noise")
axes[0].plot(range(len(best_high_noise)), best_high_noise, "-o", linewidth=2.5, label="High noise")
axes[0].set_title("Effect of observation noise on 2D BO", fontsize=16, fontweight="bold")
axes[0].set_xlabel("BO iteration", fontsize=18, fontweight="bold")
axes[0].set_ylabel("Best observed value", fontsize=18, fontweight="bold")
axes[0].legend(prop={"size": 9, "weight": "bold"})
style_ax(axes[0])

h_low = history_low_noise[-1]
h_high = history_high_noise[-1]
cont = axes[1].contourf(X1.numpy(), X2.numpy(), Y_grid_2d.numpy(), levels=30)
axes[1].scatter(
    h_low["train_X"][:, 0].numpy(),
    h_low["train_X"][:, 1].numpy(),
    s=55,
    marker="o",
    color="white",
    edgecolors="black",
    zorder=3,
    label="Low noise",
)
axes[1].scatter(
    h_high["train_X"][:, 0].numpy(),
    h_high["train_X"][:, 1].numpy(),
    s=60,
    marker="s",
    facecolors="none",
    edgecolors="black",
    linewidths=1.2,
    zorder=4,
    label="High noise",
)
axes[1].set_title("Final sampling pattern: \n low vs high noise", fontsize=16, fontweight="bold")
axes[1].set_xlabel(r"$x_1$", fontsize=18, fontweight="bold")
axes[1].set_ylabel(r"$x_2$", fontsize=18, fontweight="bold")
axes[1].legend(prop={"size": 9, "weight": "bold"}, loc="upper left")
style_ax(axes[1])
plt.colorbar(cont, ax=axes[1])

plt.tight_layout()
plt.show()

## 8. Running two BO loops with different acquisition-optimisation settings

We now compare two BO runs that differ only in **how carefully the acquisition function itself is optimised**.

This is an important practical comparison because, in BoTorch, Bayesian Optimisation is really a two-level optimisation process:

1. the GP surrogate is fit to the observed data
2. the acquisition function built from that surrogate is then **numerically optimised** to choose the next candidate point

So even if the surrogate model is the same, the BO trajectory can still change depending on how thoroughly the acquisition optimisation step is carried out.

---

### What we are comparing

The two BO runs in this cell are identical in every respect except for:

- `num_restarts`
- `raw_samples`

In the first run, we use:

```python
num_restarts=3
raw_samples=15
```

This is the **weaker acquisition-optimisation** setting.

In the second run, we use:

```python
num_restarts=15
raw_samples=80
```

This is the **stronger acquisition-optimisation** setting.

Everything else is kept fixed:

- the same 2D objective
- the same initial dataset
- the same BO bounds
- the same transforms
- the same number of BO iterations
- and the same acquisition function (`LogExpectedImprovement`)

So any later difference in BO behaviour can be attributed mainly to how thoroughly the acquisition function is being optimised.

---

### What `raw_samples` means

Before BoTorch runs gradient-based local optimisation of the acquisition function, it first draws a set of **random trial points** from the search domain.

That is what `raw_samples` controls.

So:

- `raw_samples=15` means the optimiser begins with only 15 random candidate locations
- `raw_samples=80` means it begins with 80 random candidate locations

These raw samples are used to identify promising starting points for the later local optimisation phase.

So `raw_samples` determines how much initial random coverage of the acquisition landscape is used before refinement begins.

A larger value means:

- the acquisition optimiser gets a broader initial look at the domain
- it is less likely to miss good regions of the acquisition surface
- but it also costs more computation

A smaller value means:

- faster execution
- but a greater chance that the optimiser never even starts near a genuinely strong acquisition peak

---

### What `num_restarts` means

Once the raw samples have been generated, BoTorch selects promising starting locations and then runs a local optimisation procedure from several of them.

That is what `num_restarts` controls.

So:

- `num_restarts=3` means BoTorch performs local acquisition optimisation from 3 starting points
- `num_restarts=15` means it performs local acquisition optimisation from 15 starting points

Each restart is effectively another attempt to climb the acquisition surface from a different initial location.

This matters because acquisition functions in BO are often **non-convex**.
That means they can have:

- multiple local maxima
- flat regions
- irregular geometry
- and local optimisation traps

So `num_restarts` is one of the main tools used to reduce the risk of settling for a poor local optimum of the acquisition function.

A larger number of restarts means:

- a more thorough search of the acquisition landscape
- a better chance of identifying a stronger candidate point
- but also a higher computational cost

A smaller number of restarts means:

- faster optimisation
- but a greater chance that the next BO point is only locally good rather than globally strong under the acquisition rule

---

### Why these two parameters matter together

These two settings work together:

- `raw_samples` controls the quality of the **initial candidate pool**
- `num_restarts` controls how many of those promising initial points are then **refined locally**

So the pair

```python
num_restarts=3
raw_samples=15
```

corresponds to a relatively lightweight and coarse acquisition-optimisation procedure.

By contrast, the pair

```python
num_restarts=15
raw_samples=80
```

corresponds to a much more thorough search of the acquisition landscape.

This makes the comparison very practical, because it isolates a real trade-off in BoTorch workflows:

> **faster acquisition optimisation versus more careful acquisition optimisation**

---

### Why this is a meaningful practical BO comparison

It is easy to think of BO as if the next point were determined only by the surrogate and the acquisition function definition.

But in practice, the next point also depends on how successfully we optimise the acquisition function itself.

So this section is asking:

> **How much does the quality of acquisition optimisation affect the actual BO trajectory?**

This is especially relevant in 2D and higher dimensions, where the acquisition surface is already rich enough to make local optimisation nontrivial.

---

### Why `store_posterior=True` is used here

As in the earlier sections, both runs set:

- `store_posterior=True`
- `grid_shape=X1.shape`

This keeps the BO history information-rich and makes it possible to inspect not only best-value trajectories but also the resulting spatial behaviour of the BO loop if needed.

So although the main point of this section is acquisition optimisation quality, we still preserve the full stored history for downstream analysis.

---

### What to expect before seeing the results

Before plotting anything, we can anticipate two possibilities.

A stronger acquisition-optimisation setting may:

- identify better candidate points more reliably
- produce faster improvement in the best observed value
- and lead to a more focused or more efficient search trajectory

But the result is not always guaranteed to be straightforward.

Because BO is sequential, even a slightly different early candidate can change the later surrogate and therefore the whole future search path.

So this comparison is not just about “better numerical settings always win.”
It is about how acquisition-optimisation quality interacts with the sequential nature of BO.

---

### Why this section matters in practice

This is one of the most useful practical lessons in BoTorch.

When BO performance looks disappointing, the issue is not always:

- the surrogate model
- or the acquisition function formula itself

Sometimes the problem is simply that the acquisition function was **not optimised carefully enough**.

So `raw_samples` and `num_restarts` are not just minor tuning knobs.
They are part of the real numerical optimisation problem that BO must solve at each iteration.

---

### Key takeaway

This cell runs two otherwise identical 2D BO loops in order to compare:

- a **weaker acquisition-optimisation setting** with fewer raw samples and fewer restarts
- and a **stronger acquisition-optimisation setting** with more raw samples and more restarts

Here:

- `raw_samples` controls how many random trial points are used to initialise the search over the acquisition surface
- `num_restarts` controls how many local optimisation runs are launched from promising starting points

Together, these parameters determine how thoroughly BoTorch searches for the next acquisition-maximising candidate point.

In [ ]:
history_weak_opt = run_bo_loop_practical(
    train_X_init=train_X_2d,
    train_Y_init=train_Y_2d,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    X_eval=X_grid_2d,
    n_steps=50,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=3,
    raw_samples=15,
    store_posterior=True,
    grid_shape=X1.shape,
)

history_strong_opt = run_bo_loop_practical(
    train_X_init=train_X_2d,
    train_Y_init=train_Y_2d,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    X_eval=X_grid_2d,
    n_steps=50,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=15,
    raw_samples=80,
    store_posterior=True,
    grid_shape=X1.shape,
)

## 9. Comparing BO performance and final sampling patterns under different acquisition-optimisation settings

Now that the two BO runs have been completed, we can compare the practical effect of **acquisition-optimisation quality** from two perspectives:

- how the **best observed value** evolves over BO iterations
- and where the optimiser ultimately chooses to **place evaluations** on the true 2D objective surface

This is a particularly interesting comparison because the result is **not** the naive one.

At first glance, one might expect that a more carefully optimised acquisition function should automatically produce a better BO trajectory.
But the attached result shows that this is not necessarily true.

---

### What the code does

The first two lines extract, from each stored BO state, the current best observed objective value:

```python
best_weak_opt = [float(torch.min(h["train_Y"])) for h in history_weak_opt]
best_strong_opt = [float(torch.min(h["train_Y"])) for h in history_strong_opt]
```

Because this is a minimisation problem, lower values are better.
So these two sequences track how quickly each BO run improves the best solution it has found so far.

The figure then creates two side-by-side panels.

#### Left panel
The left panel plots:

- the best observed value over BO iterations for the **weak acquisition-optimisation** run
- and the best observed value over BO iterations for the **stronger acquisition-optimisation** run

This gives a direct optimisation-performance comparison.

#### Right panel
The right panel uses the **true objective surface** `Y_grid_2d` as a contour background, and overlays the final sampled locations from the two BO runs:

- hollow square markers for the **weak** acquisition-optimisation run
- circular white markers for the **stronger** acquisition-optimisation run

So this panel shows how the two acquisition-optimisation settings lead to different final spatial allocations of the BO budget.

---

### How to read the left panel

The left panel tracks

```python
min(y_observed)
```

as a function of BO iteration.

This should be interpreted as the practical optimisation outcome:

- if one curve drops earlier or lower, that BO run is finding better objective values sooner
- if one curve plateaus for longer, that run is spending more time without improving the current best value

In the result, the surprising feature is that the **weaker** acquisition-optimisation run eventually finds the deeper basin first, while the **stronger** run remains trapped for much longer around a shallower value.

So the main practical message from the left panel is:

> **more careful optimisation of the acquisition function does not automatically imply better BO performance.**

---

### How to read the right panel

The right panel should be interpreted differently.

Here, the background contour is the **true hidden objective**, while the markers show the final set of points sampled by each BO run.

This answers a different question:

> **Did the two BO runs allocate their evaluations in the same way, or did the acquisition-optimisation settings lead to different search trajectories?**

In the attached result, the answer is clearly yes.

The two runs did not merely choose slightly different next points.
They ended up exploring and exploiting the search space in visibly different ways.

The **stronger** acquisition-optimisation run concentrated more heavily around the upper-right basin for a long time, while the **weaker** run produced a broader pattern that eventually allowed it to reach the deeper lower-left basin.

So the right panel makes the path-dependent effect of acquisition-optimisation settings spatially visible.

---

### Why stronger acquisition optimisation does **not** necessarily lead to better BO

This is the most important conceptual point in the section.

It is tempting to think:

> stronger acquisition optimisation = better candidate point = better BO

But that logic is incomplete.

The acquisition function is built from the **current surrogate**, and the current surrogate may still be imperfect.

So if the surrogate currently believes that a **local basin** is the most promising region, then a more carefully optimised acquisition function may simply choose points in that same misleading region more consistently.

In other words:

- stronger acquisition optimisation means we are better at maximising the acquisition function
- but it does **not** mean the acquisition function itself is pointing toward the true global optimum

It only means we are following the current surrogate’s belief more faithfully.

If that belief is still biased toward a locally attractive but globally suboptimal region, then stronger acquisition optimisation can actually produce **more persistent exploitation of the wrong basin**.

That is exactly what the attached result suggests.

---

### Why the weaker run can sometimes do better

The weaker acquisition-optimisation run uses fewer raw samples and fewer local restarts.

That means it is solving the acquisition subproblem less thoroughly.

At first that sounds purely bad, but in a sequential BO process it can sometimes have an unintended side effect:

- it may fail to identify the single strongest local acquisition maximum
- and instead select a different, slightly less “optimal” candidate under the current surrogate
- which can introduce broader effective exploration

That broader search can, in some cases, help the BO loop escape a misleading local regime earlier.

So the weaker acquisition-optimisation run is not better because coarse optimisation is inherently superior.
Rather, it can sometimes behave in a way that is **less aggressively exploitative**, and that can accidentally help when the surrogate is still wrong.

---

### What this says about the exploration–exploitation balance

This comparison highlights an important subtlety:

- stronger acquisition optimisation improves how well we solve the **acquisition subproblem**
- but BO performance depends on the quality of the **whole sequential process**, not just one optimisation step in isolation

If the acquisition function is already tilted too strongly toward exploitation of a currently favoured basin, then better numerical optimisation of that acquisition function can amplify the exploitative tendency.

So stronger acquisition optimisation can lead to:

- more accurate local decision-making
- but not necessarily better global optimisation

This is especially true in multimodal problems like this one, where the surrogate may initially underrepresent the true best basin.

---

### Why this result is pedagogically useful

The attached result is actually very valuable because it shows a more realistic BO lesson than the simplistic slogan “more optimisation is always better.”

It shows that in practice:

- BO is path dependent
- acquisition optimisation is not the same as objective optimisation
- and a more thoroughly optimised acquisition function can still produce a worse overall BO trajectory if the surrogate is currently misleading

That is a much deeper and more practical insight.

---

### Why this comparison is especially useful in 2D

In one dimension, differences in acquisition-optimisation settings may only shift the selected points slightly.

In two dimensions, those differences can be amplified into genuinely different trajectories, because the optimiser has more freedom to:

- remain trapped in one promising-looking basin
- spread more broadly
- or jump between regions depending on how the acquisition landscape is searched

So the 2D setting makes the consequences of acquisition-optimisation choices much easier to see.

---

### Key takeaway

This figure compares two BO runs that differ only in how thoroughly the acquisition function is numerically optimised.

The left panel shows the effect on **optimisation performance** through the best observed value over time.
The right panel shows the effect on **sampling behaviour** through the final point locations on the true 2D objective.

The key lesson is subtle but important:

> **stronger acquisition optimisation does not automatically lead to better Bayesian Optimisation.**

It only means that we are following the current surrogate-induced acquisition landscape more faithfully.
If that landscape is still biased toward a locally attractive but globally suboptimal region, then stronger acquisition optimisation can simply produce more persistent exploitation of the wrong basin.

In [ ]:
best_weak_opt = [float(torch.min(h["train_Y"])) for h in history_weak_opt]
best_strong_opt = [float(torch.min(h["train_Y"])) for h in history_strong_opt]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

axes[0].plot(range(len(best_weak_opt)), best_weak_opt, "-o", linewidth=2.5, label="Weak acq optimisation")
axes[0].plot(range(len(best_strong_opt)), best_strong_opt, "-o", linewidth=2.5, label="Strong acq optimisation")
axes[0].set_title("Effect of acquisition optimisation settings in 2D", fontsize=16, fontweight="bold")
axes[0].set_xlabel("BO iteration", fontsize=18, fontweight="bold")
axes[0].set_ylabel("Best observed value", fontsize=18, fontweight="bold")
axes[0].legend(prop={"size": 9, "weight": "bold"})
style_ax(axes[0])

h_weak = history_weak_opt[-1]
h_strong = history_strong_opt[-1]
cont = axes[1].contourf(X1.numpy(), X2.numpy(), Y_grid_2d.numpy(), levels=30)
axes[1].scatter(
    h_weak["train_X"][:, 0].numpy(),
    h_weak["train_X"][:, 1].numpy(),
    s=60,
    marker="s",
    facecolors="none",
    edgecolors="black",
    linewidths=1.2,
    zorder=3,
    label="Weak",
)
axes[1].scatter(
    h_strong["train_X"][:, 0].numpy(),
    h_strong["train_X"][:, 1].numpy(),
    s=55,
    marker="o",
    color="white",
    edgecolors="black",
    zorder=4,
    label="Strong",
)
axes[1].set_title("Final sampling pattern: \n weak vs strong acquisition optimisation", fontsize=16, fontweight="bold")
axes[1].set_xlabel(r"$x_1$", fontsize=18, fontweight="bold")
axes[1].set_ylabel(r"$x_2$", fontsize=18, fontweight="bold")
axes[1].legend(prop={"size": 9, "weight": "bold"}, loc="upper left")
style_ax(axes[1])
plt.colorbar(cont, ax=axes[1])

plt.tight_layout()
plt.show()

## 10. Constructing two contrasting initial designs in 2D

Before any BO iterations are run, the optimiser is already constrained by the **quality of its initial dataset**.

That makes the initial design an important practical choice.

In this section, we compare two deliberately different 2D initial designs:

- a **more space-filling initial design**
- and a **clustered initial design**

The goal is to see how much the quality of the initial information available to the surrogate affects the later BO trajectory.

---

### What the code does

The code first defines two sets of initial input locations.

#### More space-filling initial design
`train_X_good_init` places six points across a broader region of the 2D search space.

The purpose is not to make the design perfectly uniform, but to give the surrogate a wider initial view of the domain, including information from more than one part of the landscape.

#### Clustered initial design
`train_X_bad_init` places all six points close together in the lower-left region.

This creates a deliberately poor initial design in which a large portion of the search space is left almost completely unconstrained.

So the two designs differ not in the number of initial observations, but in how well those observations cover the domain.

---

### Why this is a meaningful BO comparison

Bayesian Optimisation does not begin from scratch in a completely information-free way.
It begins from the initial observed dataset.

That means the early GP surrogate is shaped strongly by:

- where the initial points are placed
- and how much of the search space they cover

A more space-filling initial design can give the surrogate:

- broader information about the domain
- better early geometric awareness
- and a better chance of recognising multiple promising regions

By contrast, a clustered design may cause the surrogate to begin with a very narrow view of the problem, leaving large regions poorly constrained.

So this comparison asks a very practical question:

> **How much does the quality of the initial design matter for later BO behaviour?**

---

### Why both designs use the same number of points

It is important that both initial designs contain exactly six points.

This means the comparison is not about having **more data** versus **less data**.

Instead, it isolates the effect of **how the same data budget is allocated initially**.

So any later difference in BO behaviour can be attributed mainly to:

- broader initial coverage
- versus clustered initial coverage

rather than to a difference in the total number of observations.

---

### What the noisy outputs are doing

After defining the input locations, the code evaluates the true objective at those points and adds a small perturbation:

```python
train_Y_good_init = objective_2d(train_X_good_init) + 0.03 * torch.rand_like(objective_2d(train_X_good_init))
train_Y_bad_init = objective_2d(train_X_bad_init) + 0.03 * torch.rand_like(objective_2d(train_X_bad_init))
```

So both initial datasets are slightly noisy, just as in the earlier sections.

This keeps the comparison realistic and consistent with the rest of the tutorial:

- the objective is not treated as perfectly observed
- and the GP must still infer the latent surface from noisy values

---

### How to read the figure

The figure shows the two initial designs on top of the **true hidden 2D objective surface**.

The contour background is the same in both panels.
Only the initial point locations change.

#### Left panel
The left panel shows the **more space-filling design**.

The points are distributed across a broader portion of the domain, so the surrogate begins with information from multiple regions of the surface.

#### Right panel
The right panel shows the **clustered design**.

The points are concentrated in one area, leaving most of the search space initially unobserved.

So this figure should be interpreted as a comparison of the **starting information state** of the BO loop.

---

### Why this matters for later BO behaviour

A BO loop is path dependent.

If the initial surrogate is better informed about the structure of the domain, then the early acquisition decisions can already be very different.

So a more space-filling initial design may help BO:

- identify promising regions earlier
- avoid overcommitting to one poorly informed area
- and build a better global picture of the objective

By contrast, a clustered design may force the BO loop to spend many early iterations simply recovering from poor initial coverage.

That is why this section is such a useful practical complement to the earlier transform, noise, and acquisition-optimisation comparisons.

---

### Why this comparison is especially useful in 2D

In one dimension, even a clustered design still gives the surrogate some sense of the global line.

In two dimensions, poor initial coverage is much more serious, because there are many more directions in which the surface can remain unconstrained.

So the effect of initial design quality becomes much easier to see in 2D:

- good coverage can help the optimiser begin with a more informative surrogate
- poor coverage can leave large regions of the search space essentially invisible at the start

That makes this an especially appropriate setting for studying initial design quality.

---

### Key takeaway

This cell defines two contrasting 2D initial datasets:

- a more space-filling design
- and a clustered design

The two designs use the same number of initial observations, so the comparison isolates the effect of **initial coverage quality** rather than data quantity.

The figure shows how differently informed the BO loop can be at the very start, depending on how the initial evaluations are placed.

In [ ]:
train_X_good_init = torch.tensor(
    [
        [-2.2, -2.0],
        [-1.2, -1.0],
        [-0.5,  0.0],
        [ 0.8,  1.2],
        [ 1.5,  1.8],
        [ 2.2,  2.2],
    ],
    dtype=torch.double,
)

train_X_bad_init = torch.tensor(
    [
        [-2.2, -2.0],
        [-2.0, -1.8],
        [-1.8, -2.2],
        [-1.6, -1.7],
        [-2.1, -1.5],
        [-1.7, -1.9],
    ],
    dtype=torch.double,
)

train_Y_good_init = objective_2d(train_X_good_init) + 0.03 * torch.rand_like(objective_2d(train_X_good_init))
train_Y_bad_init = objective_2d(train_X_bad_init) + 0.03 * torch.rand_like(objective_2d(train_X_bad_init))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), sharex=True, sharey=True)

for ax, X_init, title in zip(
    axes,
    [train_X_good_init, train_X_bad_init],
    ["More space-filling initial design", "Clustered initial design"],
):
    cont = ax.contourf(X1.numpy(), X2.numpy(), Y_grid_2d.numpy(), levels=30)
    ax.scatter(
        X_init[:, 0].numpy(),
        X_init[:, 1].numpy(),
        s=65,
        color="white",
        edgecolors="black",
        zorder=3,
    )
    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel(r"$x_1$", fontsize=18, fontweight="bold")
    ax.set_ylabel(r"$x_2$", fontsize=18, fontweight="bold")
    style_ax(ax)
    plt.colorbar(cont, ax=ax)

plt.tight_layout()
plt.show()

## 11. Running two BO loops from different initial designs

With the two contrasting initial datasets now defined, we can run the same 2D BO loop starting from each one.

This section keeps the BO procedure fixed and changes only the **initial design quality**.

So the comparison is asking:

> **If the optimiser begins from a broader, more space-filling initial dataset rather than a clustered one, how much does that affect the later BO trajectory?**

---

### What we are comparing

The two BO runs in this cell are identical in every respect except for the initial dataset:

- `train_X_good_init`, `train_Y_good_init`
- `train_X_bad_init`, `train_Y_bad_init`

In the first run, the BO loop starts from the **more space-filling initial design**.

In the second run, the BO loop starts from the **clustered initial design**.

Everything else is held fixed:

- the same 2D objective
- the same search bounds
- the same number of BO iterations
- the same use of input normalisation and output standardisation
- and the same acquisition-optimisation settings

So any later difference in BO behaviour can be attributed mainly to the quality of the starting design.

---

### Why this is a meaningful practical BO comparison

Bayesian Optimisation is highly path dependent.

The initial dataset determines the first surrogate model, and that first surrogate influences the first acquisition decision.
That first decision then changes the dataset, which changes the next surrogate, and so on.

So the quality of the initial design does not just matter for the beginning of the run.
It can influence the **entire later trajectory** of the optimisation process.

This is why initial design is such an important practical choice:

- it shapes the earliest beliefs of the GP
- it influences which regions look promising or uncertain
- and it can determine whether BO explores broadly or gets trapped in a narrow part of the domain early on

---

### What the code does

The code runs `run_bo_loop_practical(...)` twice:

- once with the more space-filling initial design
- once with the clustered initial design

Both runs use:

- `use_input_transform=True`
- `use_outcome_transform=True`

so the comparison is focused on **initial design quality**, not on preprocessing choices.

Both runs also use:

- `num_restarts=10`
- `raw_samples=50`

so the acquisition-optimisation settings are kept the same.

This makes the experiment controlled: the only substantive difference is the distribution of the initial observed points.

---

### Why `store_posterior=False` is used here

In this section, the main quantities we want to compare are:

- best observed value over BO iterations
- and final sampling pattern on the true objective surface

For that purpose, we do **not** need to store posterior grids at every BO step.

So both runs set:

- `store_posterior=False`

This makes the comparison more computationally efficient, because the BO loop no longer evaluates and stores full posterior surfaces on the dense 2D grid during every iteration.

That is especially useful here because the goal is to compare BO outcomes, not to visualise the surrogate mean at every stage.

So this is a deliberate performance choice: we keep the optimisation logic the same, but avoid unnecessary posterior-grid storage.

---

### Why `X_eval` is still passed

Although `store_posterior=False`, the call still includes:

- `X_eval=X_grid_2d`

In this case, `X_eval` is effectively inactive because posterior storage is turned off.

So it does not materially affect the computation here.

The BO loop will only use `X_eval` when `store_posterior=True`.

---

### What to expect before seeing the results

Before plotting anything, we can anticipate the qualitative difference between the two runs.

A more space-filling initial design may help BO:

- build a better early surrogate
- recognise multiple promising regions sooner
- and avoid spending too many early iterations recovering from poor initial coverage

A clustered initial design may instead cause BO to:

- begin with a narrow and locally biased surrogate
- leave large parts of the domain highly unconstrained
- and potentially spend many early iterations correcting for the poor initial placement of observations

So the comparison is really about whether a better start leads to a better overall optimisation path.

---

### Why this matters in practice

This section is especially relevant because, in many BO applications, the user has direct control over the initial batch of evaluations.

That means there is usually a real design decision to make before BO even begins:

- should the initial points be spread out across the domain?
- or can they be concentrated in a small region?

This cell sets up exactly that comparison in a controlled 2D setting.

It therefore addresses one of the most practically important BO questions in the notebook:

> **How much does the starting information quality matter?**

---

### Key takeaway

This cell runs two otherwise identical 2D BO loops in order to compare:

- a **more space-filling initial design**
- and a **clustered initial design**

The BO logic, transforms, and acquisition-optimisation settings are all held fixed.

So the rest of the section can isolate the practical effect of **initial design quality** on the later Bayesian Optimisation trajectory.

In [ ]:
history_good_init = run_bo_loop_practical(
    train_X_init=train_X_good_init,
    train_Y_init=train_Y_good_init,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    n_steps=50,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=10,
    raw_samples=50,
    X_eval=X_grid_2d,
    store_posterior=False,
)

history_bad_init = run_bo_loop_practical(
    train_X_init=train_X_bad_init,
    train_Y_init=train_Y_bad_init,
    objective_fn=objective_2d,
    bounds=bounds_2d,
    n_steps=50,
    use_input_transform=True,
    use_outcome_transform=True,
    num_restarts=10,
    raw_samples=50,
    X_eval=X_grid_2d,
    store_posterior=False,
)

## 12. Comparing BO performance and final sampling patterns under different initial designs

Now that the two BO runs have been completed, we can compare the practical effect of **initial design quality** from two perspectives:

- how the **best observed value** evolves over BO iterations
- and where the optimiser ultimately chooses to **place evaluations** on the true 2D objective surface

This is a very practically important comparison, because the initial design is one of the few parts of the BO workflow that the user often controls directly before the sequential loop even begins.

---

### What the code does

The first two lines extract, from each stored BO state, the current best observed objective value:

```python
best_good_init = [float(torch.min(h["train_Y"])) for h in history_good_init]
best_bad_init = [float(torch.min(h["train_Y"])) for h in history_bad_init]
```

Because this is a minimisation problem, lower values are better.
So these two sequences track how quickly each BO run improves the best solution it has found so far.

The figure then creates two side-by-side panels.

#### Left panel
The left panel plots:

- the best observed value over BO iterations for the **more space-filling initial design**
- and the best observed value over BO iterations for the **clustered initial design**

This gives a direct optimisation-performance comparison.

#### Right panel
The right panel uses the **true objective surface** `Y_grid_2d` as a contour background, and overlays the final sampled locations from the two BO runs:

- hollow square markers for the **clustered initial design** run
- circular white markers for the **more space-filling initial design** run

So this panel shows how the two starting designs lead to different spatial allocations of the BO budget.

---

### How to read the left panel

The left panel tracks

```python
min(y_observed)
```

as a function of BO iteration.

This should be interpreted as the practical optimisation outcome:

- if one curve drops earlier, that BO run is discovering strong regions faster
- if one curve plateaus longer, that BO run is taking more iterations before finding comparably good points

In the attached result, the **more space-filling initial design** finds strong low-value regions much earlier, while the **clustered initial design** improves more slowly and only catches up later.

So the main message from the left panel is:

> **initial design strongly affects the speed of optimisation.**

---

### How to read the right panel

The right panel should be interpreted differently.

Here, the background contour is the **true hidden objective**, while the markers show the final set of points sampled by each BO run.

This answers a different question:

> **Did the two BO runs use their evaluation budget in the same way, or did the starting design change the search trajectory?**

In the attached result, the answer is clearly yes.

The run that started from the **clustered design** had to spend many more evaluations effectively recovering from its poor initial coverage.
By contrast, the run that started from the **more space-filling design** could identify and exploit strong regions earlier.

So the right panel makes the path-dependent effect of the starting design visually explicit.

---

### What the attached result is telling us

This result is actually very instructive.

At first, the more space-filling design performs much better:

- it begins from a stronger information state
- it finds better regions earlier
- and it reaches a strong low-value basin much more quickly

The clustered design starts off much worse:

- it begins with narrow local coverage
- it must spend more BO iterations correcting for poor initial information
- and its best-value curve lags for a substantial part of the run

But as the BO loop continues, the clustered-design run eventually catches up and approaches the same final basin.

That gives the correct practical conclusion:

> **initial design often affects the speed of optimisation much more strongly than the final attainable result, provided the BO loop is given enough iterations.**

---

### Why this matters in reality

This distinction between **speed** and **eventual outcome** is very important in practical BO.

In a synthetic notebook example, running more BO iterations is usually only a matter of patience.

But in real applications, each BO step may correspond to:

- a costly laboratory experiment
- an expensive simulation
- a long measurement cycle
- or the use of scarce materials, compute, or human time

So if a better initial design allows BO to find strong candidate regions in fewer iterations, that can reduce:

- **time cost**
- **money cost**
- **experimental burden**
- and **overall optimisation risk**

In practice, this can matter far more than whether two runs eventually converge to a similar final value after a large number of steps.

So the main real-world lesson here is not just “good initial designs are nice.”
It is:

> **good initial designs can materially reduce the number of expensive BO iterations required to reach useful performance.**

---

### Why both runs can still reach similar final results

It is also important not to overstate the role of initial design.

As this figure shows, the clustered-design run is not permanently doomed.
Once BO is allowed to continue for enough iterations, it can still explore, update the surrogate, and eventually reach the same good basin.

That is why the correct conclusion is not:

> “A bad initial design makes BO fail.”

Instead, the more accurate conclusion is:

> “A bad initial design can make BO much slower and less sample-efficient, even if the optimiser can often recover given enough steps.”

That is a much more realistic and useful practical takeaway.

---

### Why this happens

Bayesian Optimisation is path dependent.

A better initial design gives the first GP surrogate:

- broader information about the domain
- better awareness of multiple regions
- and a stronger basis for early acquisition decisions

A clustered design gives the first surrogate:

- narrow local information
- poorer global awareness
- and a greater risk of spending early BO steps in the wrong place

So the initial design mainly influences the **efficiency** of the early-to-mid optimisation trajectory.

That is exactly what the left panel reveals.

---

### Why this comparison is especially useful in 2D

In one dimension, even a weak initial design often still gives the surrogate a rough sense of the overall domain.

In two dimensions, poor initial coverage is much more damaging because large parts of the surface can remain essentially invisible at the beginning.

So the contrast between:

- faster optimisation under a better initial design
- and slower recovery under a clustered design

is much easier to see in 2D.

This makes the present example a particularly good demonstration of why initial design quality matters in practice.

---

### Key takeaway

This figure compares two BO runs that differ only in the **quality of the initial design**.

The left panel shows the effect on **optimisation speed** through the best observed value over time.
The right panel shows the effect on **sampling behaviour** through the final point locations on the true 2D objective.

The main conclusion is:

> **initial design strongly affects how quickly Bayesian Optimisation reaches good regions of the search space.**

In real applications, that directly affects practical costs such as time, money, and experimental budget.

At the same time, this example also shows that with a sufficiently large number of BO steps, both runs can often still recover and approach a similar final result.
So initial design usually matters most for **sample efficiency**, not necessarily for the existence of a good eventual solution.

In [ ]:
best_good_init = [float(torch.min(h["train_Y"])) for h in history_good_init]
best_bad_init = [float(torch.min(h["train_Y"])) for h in history_bad_init]

h_good = history_good_init[-1]
h_bad = history_bad_init[-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

axes[0].plot(range(len(best_good_init)), best_good_init, "-o", linewidth=2.5, label="More space-filling initial design")
axes[0].plot(range(len(best_bad_init)), best_bad_init, "-o", linewidth=2.5, label="Clustered initial design")
axes[0].set_title("Effect of initial design quality on 2D BO", fontsize=18, fontweight="bold")
axes[0].set_xlabel("BO iteration", fontsize=22, fontweight="bold")
axes[0].set_ylabel("Best observed value", fontsize=22, fontweight="bold")
axes[0].legend(prop={"size": 9, "weight": "bold"})
style_ax(axes[0])

cont = axes[1].contourf(X1.numpy(), X2.numpy(), Y_grid_2d.numpy(), levels=30)
axes[1].scatter(
    h_bad["train_X"][:, 0].numpy(),
    h_bad["train_X"][:, 1].numpy(),
    s=60,
    marker="s",
    facecolors="none",
    edgecolors="black",
    linewidths=1.2,
    zorder=3,
    label="Clustered initial design",
)
axes[1].scatter(
    h_good["train_X"][:, 0].numpy(),
    h_good["train_X"][:, 1].numpy(),
    s=55,
    marker="o",
    color="white",
    edgecolors="black",
    zorder=4,
    label="More space-filling initial design",
)
axes[1].set_title("Final sampling pattern: \n good vs clustered initial design", fontsize=16, fontweight="bold")
axes[1].set_xlabel(r"$x_1$", fontsize=18, fontweight="bold")
axes[1].set_ylabel(r"$x_2$", fontsize=18, fontweight="bold")
axes[1].legend(prop={"size": 9, "weight": "bold"}, loc="upper left")
style_ax(axes[1])
plt.colorbar(cont, ax=axes[1])

plt.tight_layout()
plt.show()

## 🧭 Closing Remarks

In this tutorial, we moved from the **standard sequential Bayesian Optimisation loop itself** to the **practical modelling choices that shape how that loop behaves in BoTorch**.

The central idea was that standard BO is not only determined by:

- the Gaussian Process surrogate,
- the acquisition function,
- and the repeated update cycle,

but also by a set of modelling and numerical decisions that strongly affect how stable, efficient, and interpretable the optimisation process becomes in practice.

At a conceptual level, the BO workflow remained the same throughout:

1. fit a surrogate model to the currently observed data,
2. construct an acquisition function from that surrogate,
3. optimise that acquisition function to choose the next point,
4. evaluate the objective there,
5. update the dataset,
6. and repeat.

What changed across the notebook was not the structure of BO itself, but the **conditions under which that structure was run**.

That is exactly what made the comparisons useful.

Across the notebook, we examined several practical BoTorch choices:

- whether inputs are normalised,
- whether scalar outputs are standardised,
- how noisy the observations are,
- how carefully the acquisition function is numerically optimised,
- and how the quality of the initial design influences the later BO trajectory.

These are not cosmetic implementation details.
They affect how the GP interprets the data, how stable the hyperparameter fit becomes, how the acquisition landscape is searched, and how quickly BO can reach good regions of the search space.

The transform comparison showed that preprocessing matters internally as well as externally.

Using `Normalize` and `Standardize` did not simply rescale the data.
It led to more stable learned lengthscales, a smoother noise trajectory, and a BO path that was easier to interpret and often more reliable.

This made a key practical lesson concrete:

> a better-behaved surrogate usually produces a better-behaved BO loop.

The noise comparison showed that observational quality matters in a genuinely consequential way.

When the same objective was observed with a higher level of noise, the BO loop became less decisive, the optimisation improved more slowly, and the search trajectory could remain trapped away from the truly best basin for much longer.

So here, the important lesson was not merely that noise is inconvenient.
It was that:

> noisier observations can materially slow BO improvement and complicate the surrogate’s belief about the objective.

The acquisition-optimisation comparison added another subtle but important lesson.

It showed that stronger acquisition optimisation does **not automatically imply better BO performance**.

A more carefully optimised acquisition function follows the current surrogate-induced acquisition landscape more faithfully — but if the surrogate is still biased toward a locally attractive region, then that can simply produce more persistent exploitation of the wrong basin.

So the notebook made another important practical point explicit:

> acquisition optimisation is a real numerical optimisation problem in its own right, and improving that inner optimisation does not guarantee globally better BO behaviour.

The initial-design comparison then highlighted a different practical aspect of sample efficiency.

A better, more space-filling initial design did not necessarily change the eventual attainable solution, but it changed **how quickly** the optimiser reached strong regions of the search space.

That matters enormously in realistic BO settings, where each iteration may correspond to:

- experimental time,
- computational expense,
- material cost,
- or human effort.

So the practical conclusion there was that:

> initial design often affects the speed and cost of optimisation even when the eventual long-run solution may still be recoverable given enough iterations.


By the end of this notebook, we have not changed the fundamental BO loop introduced earlier.
Instead, we have made that loop more realistic by showing which practical decisions most strongly shape its behaviour in BoTorch.

That gives us a natural stopping point:

> we now know not only how to run a standard Bayesian Optimisation loop, but also how to think critically about the practical modelling choices that make that loop work well.

The next stage will naturally ask how these ideas extend further:

- to richer BO settings,
- to more advanced acquisition behaviour,
- and to more realistic optimisation problems where standard single-loop BO is only the starting point.